# Chapter 24 — Credit Risk and Default Modeling

The chapter's main worked examples, from the credit triangle to the negative basis.

| Section | What it reproduces | Example |
|---|---|---|
| 1 | The credit triangle is exact for a continuous premium; the "error" is the accrued premium | 24.1 |
| 2 | Bootstrapping a hazard curve from a spread curve | 24.2 |
| 3 | Hazard and recovery are not separately identified | 24.3 |
| 4 | Distance to default from a share price (Merton) | 24.4 |
| 5 | Most of a credit spread is not expected loss | 24.5 |
| 6 | Downgrade is the route to default | 24.6 |
| 7 | Correlation moves probability to both tails | 24.7 |
| 8 | Why a negative basis is not an arbitrage | 24.12 |

All of it is deterministic — no simulation anywhere — so every figure should
reproduce to the digit the chapter prints. Where the chapter prints one decimal
and the computation gives more, both are shown.

NumPy and SciPy.

In [1]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm, binom
from scipy.optimize import brentq, fsolve

N = norm.cdf
R, RATE, FREQ = 0.40, 0.03, 4          # recovery, discount rate, quarterly premiums


def show(label, got, printed, tol=6e-4):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %12.4f   book %10.4f   %s" % (label, got, printed, flag))


def cds_legs(lam, T=5.0, R=R, r=RATE, freq=FREQ, premium="quarterly", steps=200):
    """Risky annuity and protection leg under a flat hazard rate.

    premium = "continuous"  paid continuously, so both legs share one integral
              "quarterly"   premium at the quarter end, accrued premium paid on default
              "no-accrual"  premium at the quarter end, nothing on default
    A premium paid at the quarter end is discounted half a quarter longer than a
    continuous premium; that is what separates a quarterly contract from the
    triangle, by about r*dt/2.
    """
    n, dt = int(round(T * freq)), 1.0 / freq
    h = dt / steps
    ann = prot = 0.0
    for i in range(n):
        t0 = i * dt
        if premium != "continuous":
            ann += dt * exp(-r * (t0 + dt)) * exp(-lam * (t0 + dt))
        for k in range(steps):
            t = t0 + (k + 0.5) * h
            dens = lam * exp(-lam * t) * h           # default density on the slice
            prot += exp(-r * t) * dens
            if premium == "continuous":
                ann += exp(-r * t) * exp(-lam * t) * h
            elif premium == "quarterly":
                ann += (t - t0) * exp(-r * t) * dens
    return ann, (1 - R) * prot


print("a five-year CDS, quarterly premiums, r = %.0f%%, recovery %.0f%%"
      % (100 * RATE, 100 * R))

a five-year CDS, quarterly premiums, r = 3%, recovery 40%


## 1. The credit triangle is exact for a continuous premium — Example 24.1

$s = \lambda(1-R)$ is exact when the premium is paid continuously: the survival
probability and the discount factor appear on both legs and cancel, and **only
$(1-R)$ survives.** A real contract pays quarterly, and a premium paid at the quarter end is
discounted half a quarter longer than a continuous premium, which puts the par
spread about $r\,\Delta/2 \approx 0.4\%$ above the triangle at every hazard rate.
Dropping the accrued premium on default opens a second gap that grows with
$\lambda$ — the "approximation error" the textbook statement quotes, which is
the value of a contract feature rather than an error in the triangle.

In [2]:
print("%8s %14s %22s %20s" % ("lambda", "continuous", "quarterly, accrual", "quarterly, none"))
for lam in (0.01, 0.03, 0.06, 0.10, 0.20):
    rows = [1e4 * p / a for a, p in (cds_legs(lam, premium=m)
                                    for m in ("continuous", "quarterly", "no-accrual"))]
    print("%7.0f%% %14.2f %22.2f %20.2f" % (100 * lam, rows[0], rows[1], rows[2]))
print()
a_c, p_c = cds_legs(0.20, premium="continuous")
a_q, p_q = cds_legs(0.20, premium="quarterly")
a_n, p_n = cds_legs(0.20, premium="no-accrual")
s_c, s_q, s_n = 1e4 * p_c / a_c, 1e4 * p_q / a_q, 1e4 * p_n / a_n
print("continuous premium: exactly lambda(1-R) = %.1f bp" % (1e4 * 0.20 * (1 - R)))
print("quarterly timing costs %.2f%% of the spread; dropping the accrual costs another %.1f%%"
      % (100 * (s_q - s_c) / s_c, 100 * (s_n - s_q) / s_q))

  lambda     continuous     quarterly, accrual      quarterly, none
      1%          60.00                  60.23                60.30
      3%         180.00                 180.68               181.36
      6%         360.00                 361.35               364.08
     10%         600.00                 602.25               609.86
     20%        1200.00                1204.47              1235.17

continuous premium: exactly lambda(1-R) = 1200.0 bp
quarterly timing costs 0.37% of the spread; dropping the accrual costs another 2.5%


## 2. From a spread curve to a hazard curve — Example 24.2

A spread is an *average* over the whole life of the contract, so the hazard rate
for a later bucket must exceed that average when the curve slopes up — exactly as
a forward rate exceeds a swap rate. Applying the credit triangle to each quoted
spread is right for the first bucket and increasingly wrong after it.

In [3]:
QUOTES = [(1.0, 0.0080), (3.0, 0.0130), (5.0, 0.0170), (7.0, 0.0190), (10.0, 0.0200)]


def survival(t, knots, lams):
    """Piecewise-constant hazard: Q(t) = exp(-integral of lambda)."""
    acc, prev = 0.0, 0.0
    for kt, lam in zip(knots, lams):
        if t <= kt:
            return exp(-(acc + lam * (t - prev)))
        acc += lam * (kt - prev)
        prev = kt
    return exp(-(acc + lams[-1] * (t - prev)))


def par_spread(knots, lams, T, R=R, r=RATE, freq=FREQ):
    n, dt = int(round(T * freq)), 1.0 / freq
    ann = prot = 0.0
    for i in range(1, n + 1):
        t, t_prev = i * dt, (i - 1) * dt
        Q, Q_prev = survival(t, knots, lams), survival(t_prev, knots, lams)
        df = exp(-r * t)
        ann += dt * df * Q + 0.5 * dt * df * (Q_prev - Q)
        prot += df * (Q_prev - Q)
    return (1 - R) * prot / ann


knots, lams = [], []
for Tq, s in QUOTES:                          # bootstrap one bucket at a time
    knots.append(Tq)
    lams.append(0.02)
    lams[-1] = brentq(lambda x: (lams.__setitem__(-1, x),
                                 par_spread(knots, lams, Tq) - s)[1], 1e-8, 3.0)

print("%10s %12s %22s %22s" % ("maturity", "spread (bp)", "bootstrapped lambda", "triangle guess"))
prev = 0.0
for (Tq, s), lam in zip(QUOTES, lams):
    print("%9.0fy %12.0f %19.3f%% (%.0f-%.0fy) %19.3f%%"
          % (Tq, 1e4 * s, 100 * lam, prev, Tq, 100 * s / (1 - R)))
    prev = Tq

print()
print("cumulative default probability")
for Tq in (1, 3, 5, 7, 10):
    print("  %2dy   %6.2f%%" % (Tq, 100 * (1 - survival(Tq, knots, lams))))
print()
print("at the 3-5y bucket the ratio to the triangle guess is %.2f"
      % (lams[2] / (0.0170 / (1 - R))))

  maturity  spread (bp)    bootstrapped lambda         triangle guess
        1y           80               1.333% (0-1y)               1.333%
        3y          130               2.617% (1-3y)               2.167%
        5y          170               3.998% (3-5y)               2.833%
        7y          190               4.222% (5-7y)               3.167%
       10y          200               3.885% (7-10y)               3.333%

cumulative default probability
   1y     1.32%
   3y     6.36%
   5y    13.55%
   7y    20.55%
  10y    29.29%

at the 3-5y bucket the ratio to the triangle guess is 1.41


## 3. Hazard and recovery are not identified — Example 24.3

The market observes the **product** $\lambda(1-R)$ and nothing more. The recovery
assumption is not extracted from the spread; it is imposed on it. All four rows
price the CDS identically — and price a digital default swap, which depends on
$\lambda$ alone, across a range of more than three to one.

In [4]:
S_5Y = 0.0170
print("%10s %16s %26s" % ("recovery", "implied lambda", "5-year default probability"))
for Rx in (0.20, 0.40, 0.60, 0.80):
    lam = S_5Y / (1 - Rx)
    print("%9.0f%% %15.3f%% %25.2f%%" % (100 * Rx, 100 * lam, 100 * (1 - exp(-5 * lam))))

print()
for Rx in (0.20, 0.40, 0.60, 0.80):
    lam = S_5Y / (1 - Rx)
    ann, prot = cds_legs(lam, R=Rx, premium="continuous")            # each row priced with its own recovery
    print("  R=%2.0f%%   CDS reprices at %.1f bp   lambda %.3f%%   5y default %.2f%%"
          % (100 * Rx, 1e4 * prot / ann, 100 * lam, 100 * (1 - exp(-5 * lam))))
print()
print("all four reprice the CDS identically, which is the point")
print("a digital depends on lambda alone: ratio across the rows %.1f to one"
      % ((S_5Y / 0.2) / (S_5Y / 0.8)))
print("in default probability, which is what it actually pays on, the ratio is %.1f"
      % ((1 - exp(-5 * S_5Y / 0.2)) / (1 - exp(-5 * S_5Y / 0.8))))

  recovery   implied lambda 5-year default probability
       20%           2.125%                     10.08%
       40%           2.833%                     13.21%
       60%           4.250%                     19.14%
       80%           8.500%                     34.62%

  R=20%   CDS reprices at 170.0 bp   lambda 2.125%   5y default 10.08%
  R=40%   CDS reprices at 170.0 bp   lambda 2.833%   5y default 13.21%
  R=60%   CDS reprices at 170.0 bp   lambda 4.250%   5y default 19.14%
  R=80%   CDS reprices at 170.0 bp   lambda 8.500%   5y default 34.62%

all four reprice the CDS identically, which is the point
a digital depends on lambda alone: ratio across the rows 4.0 to one
in default probability, which is what it actually pays on, the ratio is 3.4


## 4. Distance to default from a share price — Example 24.4

Equity is a call on the firm's assets, so two equations in two unknowns recover
the asset value and asset volatility from what is observable. **Equity volatility
is about twice asset volatility**, and that factor is the equity's elasticity to the firm,
$N(d_1)V/E$: the leverage $V/E = 2.40$ scaled down by the call's delta.

Note also that Merton recovers too much: on default the bondholders take over the whole
of the firm's assets at no cost, still worth about 70% of the debt's face on average, and
the model has no bankruptcy costs to reduce that.

In [5]:
E, SIG_E, D_FACE, T_M, R_M = 40.0, 0.55, 80.0, 5.0, 0.05


def merton_system(p):
    V, sig_V = p
    d1 = (log(V / D_FACE) + (R_M + 0.5 * sig_V ** 2) * T_M) / (sig_V * sqrt(T_M))
    d2 = d1 - sig_V * sqrt(T_M)
    equity = V * N(d1) - D_FACE * exp(-R_M * T_M) * N(d2)
    return [equity - E, V * N(d1) * sig_V / equity - SIG_E]


V, SIG_V = fsolve(merton_system, [E + D_FACE * exp(-R_M * T_M), SIG_E * E / (E + D_FACE)])
d1 = (log(V / D_FACE) + (R_M + 0.5 * SIG_V ** 2) * T_M) / (SIG_V * sqrt(T_M))
d2 = d1 - SIG_V * sqrt(T_M)
debt = V - E
riskless = D_FACE * exp(-R_M * T_M)
spread = -log(debt / D_FACE) / T_M - R_M

show("implied asset value", V, 95.87, tol=3e-4)
show("implied asset volatility, %", 100 * SIG_V, 27.2, tol=3e-3)
show("distance to default", d2, 0.41, tol=2e-2)
show("risk-neutral 5y default probability, %", 100 * N(-d2), 34.26, tol=2e-3)
show("implied credit spread, bp", 1e4 * spread, 218, tol=5e-3)
print()
show("debt value", debt, 55.87, tol=3e-4)
show("riskless PV of debt", riskless, 62.30, tol=3e-4)
print("expected loss %.1f%% against a %.0f%% default probability"
      % (100 * (1 - debt / riskless), 100 * N(-d2)))
print("implied recovery is near %.0f%%"
      % (100 * (1 - (1 - debt / riskless) / N(-d2))))
print("equity vol / asset vol = %.2f, the elasticity N(d1) V/E" % (SIG_E / SIG_V))

implied asset value                                 95.8691   book    95.8700   ok
implied asset volatility, %                         27.1739   book    27.2000   ok
distance to default                                  0.4054   book     0.4100   ok
risk-neutral 5y default probability, %              34.2580   book    34.2600   ok
implied credit spread, bp                          218.0305   book   218.0000   ok

debt value                                          55.8691   book    55.8700   ok
riskless PV of debt                                 62.3041   book    62.3000   ok
expected loss 10.3% against a 34% default probability
implied recovery is near 70%
equity vol / asset vol = 2.02, the elasticity N(d1) V/E


## 5. Most of a spread is not expected loss — Example 24.5

**And the ratio is largest for high-grade issuers.** An AA or A issuer's
risk-neutral default probability is about sixteen times its historical rate, a
single-B issuer's under two and a half times; AAA is lower only because its
small-sample historical rate exceeds AA's. Whatever the excess compensates for,
it does not scale with default risk.

Use risk-neutral probabilities to price and hedge, historical ones to forecast
losses, and never mix them.

In [6]:
# five-year average cumulative default rates, S&P Global Ratings 2024 annual default study, 1981-2024
RATINGS = [("AAA", 40, 0.0034), ("AA", 55, 0.0028), ("A", 80, 0.0039),
           ("BBB", 130, 0.0136), ("BB", 300, 0.0575), ("B", 550, 0.1560)]

print("%10s %10s %14s %16s %10s" % ("rating", "spread", "historical", "risk-neutral", "ratio"))
for name, s_bp, hist in RATINGS:
    rn = 1 - exp(-5 * (s_bp / 1e4) / (1 - R))
    print("%10s %8d bp %13.2f%% %15.2f%% %9.1fx" % (name, s_bp, 100 * hist, 100 * rn, rn / hist))

print()
lam_hist = -log(1 - 0.0136) / 5
el = 1e4 * lam_hist * (1 - R)
show("BBB expected loss as a spread, bp", el, 16.4, tol=5e-3)
show("BBB residual, bp", 130 - el, 113.6, tol=5e-3)
print("so %.0f%% of the BBB spread is not expected loss" % (100 * (130 - el) / 130))

    rating     spread     historical     risk-neutral      ratio
       AAA       40 bp          0.34%            3.28%       9.6x
        AA       55 bp          0.28%            4.48%      16.0x
         A       80 bp          0.39%            6.45%      16.5x
       BBB      130 bp          1.36%           10.27%       7.5x
        BB      300 bp          5.75%           22.12%       3.8x
         B      550 bp         15.60%           36.77%       2.4x

BBB expected loss as a spread, bp                   16.4320   book    16.4000   ok
BBB residual, bp                                   113.5680   book   113.6000   ok
so 87% of the BBB spread is not expected loss


## 6. Downgrade is the route to default — Example 24.6

Compounding the one-year BBB default rate as though each year were independent
gives 1.39%. The transition matrix gives **2.89%** — a factor of 2.08. The matrix
gives more because a BBB issuer still alive in year four is disproportionately
likely to be a BB or B issuer by then, facing a much higher hazard than the 0.28%
it started with.

In [7]:
GRADES = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "D"]
M = np.array([
    [90.09, 9.01, 0.60, 0.10, 0.10, 0.10, 0.00, 0.00],
    [0.60, 90.59, 7.91, 0.60, 0.10, 0.20, 0.00, 0.00],
    [0.05, 2.10, 91.40, 5.50, 0.60, 0.20, 0.10, 0.05],
    [0.02, 0.20, 4.20, 89.90, 4.30, 0.90, 0.20, 0.28],
    [0.02, 0.06, 0.30, 5.60, 83.00, 8.50, 1.00, 1.52],
    [0.00, 0.05, 0.20, 0.40, 5.80, 82.00, 5.10, 6.45],
    [0.00, 0.00, 0.30, 0.60, 1.50, 12.00, 55.00, 30.60],
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 100.0]]) / 100.0

M5 = np.linalg.matrix_power(M, 5)              # D is absorbing, so powers are the 5-year matrix
row = M5[GRADES.index("BBB")]
print("where a BBB issuer stands after five years")
print("  " + "".join("%8s" % g for g in GRADES))
print("  " + "".join("%7.2f%%" % (100 * x) for x in row))
print()
print("still investment grade %.1f%%, fallen to junk %.1f%%, defaulted %.2f%%"
      % (100 * row[:4].sum(), 100 * row[4:7].sum(), 100 * row[7]))

naive = 1 - (1 - 0.0028) ** 5
show("naive compounding of 0.28%, %", 100 * naive, 1.39, tol=5e-3)
show("transition matrix, %", 100 * row[7], 2.89, tol=5e-3)
show("ratio", row[7] / naive, 2.08, tol=1e-2)

where a BBB issuer stands after five years
       AAA      AA       A     BBB      BB       B     CCC       D
     0.10%   1.37%  14.64%  62.17%  12.90%   5.06%   0.87%   2.89%

still investment grade 78.3%, fallen to junk 18.8%, defaulted 2.89%
naive compounding of 0.28%, %                        1.3922   book     1.3900   ok
transition matrix, %                                 2.8899   book     2.8900   ok
ratio                                                2.0758   book     2.0800   ok


## 7. Correlation moves probability to both tails — Example 24.7

**The expected loss does not move at all.** What correlation does is take
probability out of the middle and put it at *both* ends: at $\rho = 0.6$ there is a
53% chance that nothing defaults and an 8% chance that at least a fifth of the
portfolio does.

That double-tailed behavior is why a senior tranche is a bet on correlation and
almost nothing else — which is Chapter 26's whole subject.

In [8]:
def portfolio_distribution(rho, p=0.05, n=100, nodes=800):
    """One-factor Gaussian copula: integrate a conditional binomial over the factor."""
    x, wq = np.polynomial.legendre.leggauss(nodes)
    z = 8.0 * x                                 # the factor, truncated at 8 sigma
    w = 8.0 * wq * norm.pdf(z)
    c = norm.ppf(p)
    pz = N((c - sqrt(rho) * z) / sqrt(1 - rho)) if rho > 0 else np.full_like(z, p)
    pz = np.clip(pz, 1e-15, 1 - 1e-15)
    k = np.arange(n + 1)
    return (binom.pmf(k[None, :], n, pz[:, None]) * w[:, None]).sum(axis=0)


print("one hundred issuers, each with a 5% default probability")
print("%8s %12s %12s %18s %16s" % ("rho", "expected", "std dev", "P(no defaults)", "P(20 or more)"))
k = np.arange(101)
for rho in (0.00, 0.05, 0.15, 0.30, 0.60):
    pmf = portfolio_distribution(rho)
    mean = float((k * pmf).sum())
    sd = sqrt(float((k ** 2 * pmf).sum()) - mean ** 2)
    print("%8.2f %12.2f %12.2f %17.2f%% %15.2f%%"
          % (rho, mean, sd, 100 * pmf[0], 100 * pmf[20:].sum()))

one hundred issuers, each with a 5% default probability
     rho     expected      std dev     P(no defaults)    P(20 or more)


    0.00         5.00         2.18              0.59%            0.00%


    0.05         5.00         3.22              3.13%            0.11%


    0.15         5.00         4.89             10.86%            1.85%


    0.30         5.00         7.12             24.62%            4.91%


    0.60         5.00        11.56             53.02%            7.87%


## 8. Why a negative basis is not an arbitrage — Example 24.12

Buy the bond, buy protection, collect the basis — except the bond must be
financed, and a corporate bond does not repo at the risk-free rate. **The trade
loses money**, and it consumes balance sheet on top.

So the negative basis is not an arbitrage; it is a measure of what balance sheet
costs, and it persists exactly as long as the funding cost exceeds it.

In [9]:
bond_spread, cds_spread, funding = 220.0, 170.0, 60.0
basis = cds_spread - bond_spread
show("the basis, bp", basis, -50.0)
show("carry before funding, bp", bond_spread - cds_spread, 50.0)
show("net of funding, bp", bond_spread - cds_spread - funding, -10.0)
print()
print("the trade loses %.0f bp, and consumes balance sheet on top"
      % abs(bond_spread - cds_spread - funding))
print("it turns a profit only if funding falls below %.0f bp" % (bond_spread - cds_spread))

the basis, bp                                      -50.0000   book   -50.0000   ok
carry before funding, bp                            50.0000   book    50.0000   ok
net of funding, bp                                 -10.0000   book   -10.0000   ok

the trade loses 10 bp, and consumes balance sheet on top
it turns a profit only if funding falls below 50 bp
